In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import joblib
import os

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
X_train_scaled = pd.read_csv("../data/processed/X_train_scaled.csv")
X_test_scaled = pd.read_csv("../data/processed/X_test_scaled.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

X_train_smote = pd.read_csv("../data/processed/X_train_smote.csv")
y_train_smote = pd.read_csv("../data/processed/y_train_smote.csv").squeeze()

print("Processed data loaded successfully.")

Processed data loaded successfully.


In [3]:
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)
print("y_train       :", y_train.shape)
print("y_test        :", y_test.shape)
print("X_train_smote :", X_train_smote.shape)
print("y_train_smote :", y_train_smote.shape)

X_train_scaled: (226980, 30)
X_test_scaled : (56746, 30)
y_train       : (226980,)
y_test        : (56746,)
X_train_smote : (453204, 30)
y_train_smote : (453204,)


In [4]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc
    }


In [5]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_smote, y_train_smote)

logistic_results = evaluate_model(logistic_model, X_test_scaled, y_test)

logistic_results

{'Accuracy': 0.9736721531033025,
 'Precision': 0.053035143769968054,
 'Recall': 0.8736842105263158,
 'F1 Score': 0.1,
 'ROC-AUC': 0.9618957810936585}

In [6]:
decision_tree_model = DecisionTreeClassifier(
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

decision_tree_model.fit(X_train_scaled, y_train)

decision_tree_results = evaluate_model(decision_tree_model, X_test_scaled, y_test)

decision_tree_results

{'Accuracy': 0.9953124449300391,
 'Precision': 0.23197492163009403,
 'Recall': 0.7789473684210526,
 'F1 Score': 0.357487922705314,
 'ROC-AUC': 0.8886397694470948}

In [7]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

random_forest_model.fit(X_train_scaled, y_train)

random_forest_results = evaluate_model(random_forest_model, X_test_scaled, y_test)

random_forest_results

{'Accuracy': 0.9993832164381631,
 'Precision': 0.8846153846153846,
 'Recall': 0.7263157894736842,
 'F1 Score': 0.7976878612716763,
 'ROC-AUC': 0.9632337051698814}

In [8]:
results_df = pd.DataFrame({
    "Logistic Regression": logistic_results,
    "Decision Tree": decision_tree_results,
    "Random Forest": random_forest_results
}).T

results_df

,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Logistic Regression,0.973672,0.053035,0.873684,0.100000,0.961896
Decision Tree,0.995312,0.231975,0.778947,0.357488,0.888640
Random Forest,0.999383,0.884615,0.726316,0.797688,0.963234


In [9]:
results_df.sort_values(by="F1 Score", ascending=False)

,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Random Forest,0.999383,0.884615,0.726316,0.797688,0.963234
Decision Tree,0.995312,0.231975,0.778947,0.357488,0.888640
Logistic Regression,0.973672,0.053035,0.873684,0.100000,0.961896


In [10]:
os.makedirs("../models", exist_ok=True)

joblib.dump(logistic_model, "../models/logistic_regression.pkl")
joblib.dump(decision_tree_model, "../models/decision_tree.pkl")
joblib.dump(random_forest_model, "../models/random_forest.pkl")

print("Models saved successfully.")

Models saved successfully.


In [11]:
best_model_name = results_df["F1 Score"].idxmax()
best_model_name

'Random Forest'

In [12]:
if best_model_name == "Logistic Regression":
    best_model = logistic_model
elif best_model_name == "Decision Tree":
    best_model = decision_tree_model
else:
    best_model = random_forest_model

joblib.dump(best_model, "../models/best_model.pkl")

print(f"Best model saved: {best_model_name}")

Best model saved: Random Forest
